# Data Loader — Cross-Country Sovereign Yield Panel

Ingests US (FRED), UK (BoE spot), Germany (Bundesbank), France (Banque de France), Italy/Japan/Canada (investing.com); handles per-source quirks (German filename swap, French decimal comma, per-file date conventions); aligns to the common-trading-day intersection; builds the basis-point daily-changes panel and screens for benchmark-roll outliers.

**Script notes**

```
IB99L0 Financial Analytics - Individual Assignment
Data loader for cross-country sovereign yield PCA.

Builds two clean daily panels from heterogeneous sources:
  - LEVELS  (yield, % p.a.)
  - CHANGES (first difference of yield, basis points)

Sources & quirks handled:
  US      FRED CMT par yields        .xlsx, sheet 'Daily', cols [observation_date, DGSxx], '.' = missing
  UK      BoE zero-coupon spot       .xlsx, sheet '4. spot curve', wide matrix, maturity labels in a header row
  Germany Bundesbank current-bond    .csv  SDMX, cols TIME_PERIOD/OBS_VALUE, '.' = missing
          *** FILENAMES ARE SWAPPED *** -> mapped by internal BBK_TITLE_ENG, not filename
  France  BdF constant-maturity      .csv  ';'-delimited, decimal comma, 6 metadata header rows, wide
  Italy   investing.com YTM benchmark.csv  Date,Price,...  MM/DD/YYYY
  Japan   investing.com YTM benchmark.csv  Date,Price,...  MM/DD/YYYY
  Canada  investing.com YTM benchmark.csv  *** 10y = MM/DD/YYYY, 30y = DD/MM/YYYY ***  (explicit per file)
```

> Paths assume this notebook sits in a folder with a `./data/` subfolder holding the raw downloads, and that earlier steps have written their intermediate CSVs into the same folder. Run `load_yields` first.

In [1]:
import pandas as pd
import numpy as np
import openpyxl

In [2]:
UP = "./data/"

In [3]:
# ----------------------------------------------------------------------
# US  (FRED .xlsx)
# ----------------------------------------------------------------------
def load_fred(path, colname):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb["Daily"]
    rows = list(ws.iter_rows(min_row=2, values_only=True))  # skip header row
    dates, vals = [], []
    for d, v in rows:
        if d is None:
            continue
        dates.append(pd.Timestamp(d).normalize())
        vals.append(np.nan if (v in (None, ".", "")) else float(v))
    s = pd.Series(vals, index=pd.DatetimeIndex(dates), name=colname)
    return s[~s.index.duplicated()].sort_index()

In [4]:
# ----------------------------------------------------------------------
# UK  (BoE '4. spot curve' wide matrix)  -> pick exact maturities
# ----------------------------------------------------------------------
def load_boe(path, want_maturities=(2.0, 5.0, 10.0, 30.0)):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb["4. spot curve"]
    rows = list(ws.iter_rows(values_only=True))
    # find the header row whose first cell == 'years:'
    hdr_idx = next(i for i, r in enumerate(rows) if r and str(r[0]).strip().lower() == "years:")
    header = rows[hdr_idx]
    # map maturity(float) -> column index
    mat_col = {}
    for j, c in enumerate(header):
        if j == 0 or c is None:
            continue
        try:
            mat_col[float(c)] = j
        except (TypeError, ValueError):
            pass
    out = {}
    for m in want_maturities:
        if m not in mat_col:
            raise KeyError(f"UK maturity {m}y not found in BoE sheet")
        out[m] = mat_col[m]
    # data rows: after header; skip any non-date / blank first cells
    recs = {f"UK_{int(m)}y": [] for m in want_maturities}
    idx = []
    for r in rows[hdr_idx + 1:]:
        if not r or r[0] is None:
            continue
        try:
            dt = pd.Timestamp(r[0]).normalize()
        except (TypeError, ValueError):
            continue
        idx.append(dt)
        for m in want_maturities:
            v = r[out[m]]
            recs[f"UK_{int(m)}y"].append(np.nan if v in (None, "", ".") else float(v))
    df = pd.DataFrame(recs, index=pd.DatetimeIndex(idx))
    return df[~df.index.duplicated()].sort_index()

In [5]:
# ----------------------------------------------------------------------
# Germany (Bundesbank SDMX csv) - map by internal title, files are swapped
# ----------------------------------------------------------------------
def load_bundesbank(path):
    df = pd.read_csv(path, dtype=str)
    title = df["BBK_TITLE_ENG"].dropna().iloc[0]
    tenor = "10y" if "10 year" in title else ("30y" if "30 year" in title else "??")
    d = pd.to_datetime(df["TIME_PERIOD"], format="%Y-%m-%d", errors="coerce")
    v = pd.to_numeric(df["OBS_VALUE"].replace(".", np.nan), errors="coerce")
    s = pd.Series(v.values, index=d, name=f"DE_{tenor}").dropna(how="all")
    s = s[~s.index.duplicated()].sort_index()
    return tenor, s

In [6]:
# ----------------------------------------------------------------------
# France (Banque de France ';' csv, decimal comma, metadata rows)
# ----------------------------------------------------------------------
def load_france(path, want=("10 ans", "30 ans")):
    raw = pd.read_csv(path, sep=";", dtype=str, header=None, encoding="utf-8-sig")
    # row 0 = 'Titre :' + maturity labels ; find which columns hold our maturities
    titles = raw.iloc[0].tolist()
    col_for = {}
    for lab in want:
        # match e.g. 'Taux de l'Echeance Constante - 10 ans' ending in '10 ans' (exact token)
        for j, t in enumerate(titles):
            if isinstance(t, str) and t.strip().endswith(lab):
                col_for[lab] = j
                break
    # find first data row: first cell parses as a date
    data_start = None
    for i in range(len(raw)):
        try:
            pd.Timestamp(raw.iloc[i, 0]); data_start = i; break
        except (ValueError, TypeError):
            continue
    body = raw.iloc[data_start:].copy()
    idx = pd.to_datetime(body.iloc[:, 0], format="%Y-%m-%d", errors="coerce")
    out = {}
    for lab, j in col_for.items():
        tenor = "10y" if "10" in lab else "30y"
        col = body.iloc[:, j].str.replace(",", ".", regex=False)
        col = col.replace({"": np.nan, "-": np.nan, "NaN": np.nan, "ND": np.nan})
        out[f"FR_{tenor}"] = pd.to_numeric(col, errors="coerce").values
    df = pd.DataFrame(out, index=idx).dropna(how="all")
    return df[~df.index.duplicated()].sort_index()

In [7]:
# ----------------------------------------------------------------------
# investing.com (Italy, Japan, Canada) - explicit date format PER FILE
# ----------------------------------------------------------------------
def load_investing(path, name, date_fmt=None):
    df = pd.read_csv(path, dtype=str)
    raw = df["Date"].str.strip('"').str.strip()
    # auto-detect convention if not forced
    if date_fmt is None:
        if raw.iloc[0].count("-") == 2:
            date_fmt = "%Y-%m-%d"
        else:
            parts = raw.str.split("/", expand=True).astype(int)
            if (parts[0] > 12).any():
                date_fmt = "%d/%m/%Y"        # some first field >12 => day-first
            elif (parts[1] > 12).any():
                date_fmt = "%m/%d/%Y"        # some second field >12 => month-first
            else:
                raise ValueError(f"{name}: date convention ambiguous, set explicitly")
    d = pd.to_datetime(raw, format=date_fmt, errors="coerce")
    v = pd.to_numeric(df["Price"].str.replace('"', "", regex=False), errors="coerce")
    s = pd.Series(v.values, index=d, name=name).dropna()
    n_bad = d.isna().sum()
    print(f"   {name}: fmt={date_fmt}  n={len(s)}  "
          f"{s.index.min().date()}->{s.index.max().date()}"
          + (f"  !{n_bad} bad dates" if n_bad else ""))
    return s[~s.index.duplicated()].sort_index()

In [8]:
# ======================================================================
# LOAD EVERYTHING
# ======================================================================
print("Loading US (FRED)...")
us = pd.concat([
    load_fred(UP+"US_DGS2.xlsx",  "US_2y"),
    load_fred(UP+"US_DGS5.xlsx",  "US_5y"),
    load_fred(UP+"US_DGS10.xlsx", "US_10y"),
    load_fred(UP+"US_DGS30.xlsx", "US_30y"),
], axis=1)
print("   US range", us.index.min().date(), "->", us.index.max().date(), us.shape)

Loading US (FRED)...
   US range 2021-01-04 -> 2026-06-02 (1412, 4)


In [9]:
print("Loading UK (BoE spot)...")
uk = pd.concat([
    load_boe(UP+"UK_GLC_Nominal_daily_data_2016_to_2024.xlsx"),
    load_boe(UP+"UK_GLC_Nominal_daily_data_2025_to_present.xlsx"),
]).sort_index()
uk = uk[~uk.index.duplicated()]
print("   UK range", uk.index.min().date(), "->", uk.index.max().date(), uk.shape)

Loading UK (BoE spot)...
   UK range 2016-01-01 -> 2026-05-29 (2716, 4)


In [10]:
print("Loading Germany (Bundesbank, by internal title)...")
de = {}
for fn in ["10_year_german.csv", "30_year_german.csv"]:
    tenor, s = load_bundesbank(UP+fn)
    print(f"   file {fn:24s} -> actually {tenor}")
    de[f"DE_{tenor}"] = s
de = pd.concat(de.values(), axis=1)

Loading Germany (Bundesbank, by internal title)...
   file 10_year_german.csv       -> actually 30y
   file 30_year_german.csv       -> actually 10y


In [11]:
print("Loading France (BdF)...")
fr = load_france(UP+"all_france_bond_maturities.csv")
print("   FR range", fr.index.min().date(), "->", fr.index.max().date(), list(fr.columns))

Loading France (BdF)...
   FR range 2004-11-03 -> 2026-06-04 ['FR_10y', 'FR_30y']


In [12]:
print("Loading investing.com (Italy/Japan MM/DD/YYYY, Canada per-file)...")
it = pd.concat([
    load_investing(UP+"Italy_10-Year_Bond_Yield_Historical_Data.csv", "IT_10y"),
    load_investing(UP+"Italy_30-Year_Bond_Yield_Historical_Data.csv", "IT_30y"),
], axis=1)
jp = pd.concat([
    load_investing(UP+"Japan_10-Year_Bond_Yield_Historical_Data.csv", "JP_10y"),
    load_investing(UP+"Japan_30-Year_Bond_Yield_Historical_Data.csv", "JP_30y"),
], axis=1)
ca = pd.concat([
    load_investing(UP+"Canada_10-Year_Bond_Yield_Historical_Data.csv", "CA_10y"),
    load_investing(UP+"Canada_30-Year_Bond_Yield_Historical_Data.csv", "CA_30y"),
], axis=1)

Loading investing.com (Italy/Japan MM/DD/YYYY, Canada per-file)...
   IT_10y: fmt=%d/%m/%Y  n=1468  2021-01-01->2026-06-04
   IT_30y: fmt=%m/%d/%Y  n=1464  2021-01-01->2026-06-04
   JP_10y: fmt=%d/%m/%Y  n=1555  2021-01-03->2026-06-04
   JP_30y: fmt=%d/%m/%Y  n=1351  2021-01-04->2026-06-04
   CA_10y: fmt=%m/%d/%Y  n=1351  2021-01-04->2026-06-04
   CA_30y: fmt=%Y-%m-%d  n=1350  2021-01-04->2026-06-04


In [13]:
# ======================================================================
# ASSEMBLE PANELS
# ======================================================================
levels = pd.concat([us, uk, de, fr, it, jp, ca], axis=1).sort_index()

In [14]:
# restrict to main analysis window
WIN = ("2024-01-01", "2026-05-30")
levels = levels.loc[WIN[0]:WIN[1]]

In [15]:
# keep weekdays, forward-fill single missing prints sparingly (max 1 day) then drop rows still all-NaN
levels = levels[levels.index.dayofweek < 5]

In [16]:
print("\n=== LEVELS panel (raw, window", WIN, ") ===")
print("shape", levels.shape)
print("columns:", list(levels.columns))
print("\nMissing values per column:")
print(levels.isna().sum().to_string())
print("\nCoverage (first/last valid) per column:")
for c in levels.columns:
    s = levels[c].dropna()
    print(f"   {c:7s} {s.index.min().date()} -> {s.index.max().date()}  n={len(s)}")


=== LEVELS panel (raw, window ('2024-01-01', '2026-05-30') ) ===
shape (630, 18)
columns: ['US_2y', 'US_5y', 'US_10y', 'US_30y', 'UK_2y', 'UK_5y', 'UK_10y', 'UK_30y', 'DE_30y', 'DE_10y', 'FR_10y', 'FR_30y', 'IT_10y', 'IT_30y', 'JP_10y', 'JP_30y', 'CA_10y', 'CA_30y']

Missing values per column:
US_2y     28
US_5y     28
US_10y    28
US_30y    28
UK_2y     21
UK_5y     21
UK_10y    21
UK_30y    21
DE_30y    20
DE_10y    20
FR_10y    16
FR_30y    16
IT_10y     6
IT_30y     6
JP_10y    16
JP_30y    59
CA_10y    28
CA_30y    29

Coverage (first/last valid) per column:
   US_2y   2024-01-02 -> 2026-05-29  n=602
   US_5y   2024-01-02 -> 2026-05-29  n=602
   US_10y  2024-01-02 -> 2026-05-29  n=602
   US_30y  2024-01-02 -> 2026-05-29  n=602
   UK_2y   2024-01-02 -> 2026-05-29  n=609
   UK_5y   2024-01-02 -> 2026-05-29  n=609
   UK_10y  2024-01-02 -> 2026-05-29  n=609
   UK_30y  2024-01-02 -> 2026-05-29  n=609
   DE_30y  2024-01-02 -> 2026-05-29  n=610
   DE_10y  2024-01-02 -> 2026-05-29  n=610

In [17]:
levels.to_csv("./levels_raw.csv")
print("\nSaved ./levels_raw.csv")


Saved ./levels_raw.csv


In [18]:
# ======================================================================
# ALIGNMENT (intersection) + CHANGES PANEL (basis points) + OUTLIER SCREEN
# ======================================================================
print("\n" + "="*60)
print("BUILDING ALIGNED PANELS")
print("="*60)


BUILDING ALIGNED PANELS


In [19]:
# Intersection: keep only dates where EVERY series trades
aligned = levels.dropna(how="any")
print(f"\nIntersection alignment: {aligned.shape[0]} common trading days "
      f"({aligned.index.min().date()} -> {aligned.index.max().date()})")
print(f"Rows dropped vs union: {levels.shape[0] - aligned.shape[0]}")


Intersection alignment: 520 common trading days (2024-01-04 -> 2026-05-29)
Rows dropped vs union: 110


In [20]:
# Daily changes in BASIS POINTS (yields are in %, so *100)
changes_bp = aligned.diff().dropna() * 100.0
print(f"Changes panel (bp): {changes_bp.shape[0]} obs x {changes_bp.shape[1]} series")

Changes panel (bp): 519 obs x 18 series


In [21]:
# ---- benchmark-roll / outlier screen: flag |z| > 5 (do NOT auto-delete) ----
print("\n--- Outlier screen on daily changes (|z| > 5 sigma) ---")
z = (changes_bp - changes_bp.mean()) / changes_bp.std()
flags = []
for col in changes_bp.columns:
    hits = z.index[z[col].abs() > 5]
    for dt in hits:
        flags.append((col, dt.date(), round(changes_bp.loc[dt, col], 1), round(z.loc[dt, col], 1)))
if flags:
    fl = pd.DataFrame(flags, columns=["series", "date", "change_bp", "z"]).sort_values("z", key=lambda s: s.abs(), ascending=False)
    print(fl.to_string(index=False))
else:
    print("none beyond 5 sigma")


--- Outlier screen on daily changes (|z| > 5 sigma) ---
series       date  change_bp    z
JP_30y 2026-01-20       36.2  8.2
 UK_2y 2026-03-19       28.3  5.4
JP_10y 2025-04-04      -16.1 -5.2


In [22]:
# ---- summary stats for the methodology table ----
print("\n--- Daily change summary (bp) ---")
desc = changes_bp.agg(["mean", "std", "min", "max"]).T
desc["ann_vol_bp"] = changes_bp.std() * np.sqrt(252)
print(desc.round(2).to_string())


--- Daily change summary (bp) ---
        mean   std    min    max  ann_vol_bp
US_2y  -0.08  5.99 -28.00  23.00       95.11
US_5y   0.03  6.10 -22.00  24.00       96.90
US_10y  0.09  5.61 -19.00  19.00       89.07
US_30y  0.17  5.22 -17.00  17.00       82.89
UK_2y   0.03  5.22 -24.84  28.33       82.91
UK_5y   0.15  5.44 -22.69  20.12       86.38
UK_10y  0.21  5.53 -20.81  18.01       87.72
UK_30y  0.24  5.48 -16.98  19.58       86.94
DE_30y  0.23  4.26 -14.00  20.00       67.61
DE_10y  0.17  4.60 -16.00  22.00       72.97
FR_10y  0.20  4.80 -13.00  21.00       76.16
FR_30y  0.25  4.52 -15.00  15.00       71.76
IT_10y -0.02  5.67 -27.30  27.60       89.96
IT_30y  0.01  5.00 -19.80  19.00       79.32
JP_10y  0.39  3.15 -16.10  15.90       49.98
JP_30y  0.43  4.36 -15.70  36.20       69.20
CA_10y  0.03  5.43 -16.00  17.40       86.21
CA_30y  0.12  4.88 -15.80  16.10       77.43


In [23]:
aligned.to_csv("./levels_aligned.csv")
changes_bp.to_csv("./changes_bp.csv")
print("\nSaved ./levels_aligned.csv and ./changes_bp.csv")


Saved ./levels_aligned.csv and ./changes_bp.csv
